# Dataset subsampling

This notebook can be used to subsample a dataset pickle to debug training runs, especially for new archictures by attempting to train the architecture on a single example.

In [1]:
import sys
import os
import pickle
import gc
import copy
import numpy as np

root_dir = os.path.join(
    os.getcwd(),
    '..',
)

sys.path.append(root_dir)

In [2]:
# Load the existing large dataset

with open(os.path.join(root_dir, 'rf2aa/dataset_20240318.pkl'), 'rb') as fh:
    dataset_clean = pickle.load(fh)

In [7]:
# Define filters

def get_index_passing_filter(train_dict, f):
    for i, row in train_dict.iterrows():
        if f(row):
            return i
    raise Exception('No index passing filter')

def get_short_single_ligand(row):
    if row['LEN_EXIST'] > 100:
        return False
    if len(row['PARTNERS']) != 1:
        return False
    return True

ii = get_index_passing_filter(dataset_clean['train_dict']['sm_compl'], get_short_single_ligand)
dici = dataset_clean['train_dict']['sm_compl'].loc[ii]
print(f'{ii=} {len(dici["PARTNERS"])=} {dici["LEN_EXIST"]=}')

ii=1154 len(dici["PARTNERS"])=1 dici["LEN_EXIST"]=65


In [4]:
# Subsample the dataset

def subsample_v2(p, n_per_dataset=1, filter_by_dataset=dict()):
    assert n_per_dataset==1
    train_ID_dict = p['train_ID_dict']
    valid_ID_dict = p['valid_ID_dict']
    weights_dict = p['weights_dict']
    train_dict = p['train_dict']
    valid_dict = p['valid_dict']
    homo = p['homo']
    chid2hash = p['chid2hash']
    chid2taxid = p['chid2taxid']
    chid2smpartners = p['chid2smpartners']

    all_chain_ids = set()
    datasets = set(train_ID_dict.keys())
    
    for k, v in train_ID_dict.items():
        if k in datasets:
            v = v[:n_per_dataset]
            dic = train_dict[k]
            row_filter = filter_by_dataset.get(k, lambda x: True)
            i = [get_index_passing_filter(train_dict[k], row_filter)]
            dic = dic.loc[i]
            v = np.array(dic.loc[i]['CLUSTER'])
            
        else:
            v = []
            dic = train_dict[k][0:0]
            weights_dict[k] = []
        if 'CHAINID' in dic:
            chain_ids = dic['CHAINID']
            for ch_id in chain_ids:
                all_chain_ids.add(ch_id)
                for ch_id in ch_id.split(':'):
                    all_chain_ids.add(ch_id)

        train_ID_dict[k] = v
        train_dict[k] = dic
        weights_dict[k] = weights_dict[k][:n_per_dataset]
    
    for k, v in valid_ID_dict.items():
        v = v[:n_per_dataset]
        dic = valid_dict[k]
        dic = dic[dic.CLUSTER.isin(v)].reset_index(drop=True)
        if 'CHAINID' in dic:
            chain_ids = dic['CHAINID']
            for ch_id in chain_ids:
                all_chain_ids.add(ch_id)
                for ch_id in ch_id.split(':'):
                    all_chain_ids.add(ch_id)

        valid_ID_dict[k] = v
        valid_dict[k] = dic
    
    homo = homo.sample(n_per_dataset)
    chid2hash = {k:v for k,v in chid2hash.items() if k in all_chain_ids}
    chid2taxid = {k:v for k,v in chid2taxid.items() if k in all_chain_ids}
    chid2smpartners = {k:v for k,v in chid2smpartners.items() if k in all_chain_ids}
    return dict(
        train_ID_dict=train_ID_dict,
        valid_ID_dict=valid_ID_dict,
        weights_dict=weights_dict,
        train_dict=train_dict,
        valid_dict=valid_dict,
        homo=homo,
        chid2hash=chid2hash,
        chid2taxid=chid2taxid,
        chid2smpartners=chid2smpartners,
    )

dataset = copy.deepcopy(dataset_clean)
dataset_subsampled = subsample_v2(dataset, filter_by_dataset={'sm_compl': get_short_single_ligand})

In [5]:
# Override the train dict with the valid dict
def use_train_as_valid(dataset):
    dataset['valid_ID_dict'] = {k: [] for k,v in dataset['valid_ID_dict'].items()}
    dataset['valid_ID_dict'].update(dataset['train_ID_dict'])
    dataset['valid_dict'] = {k: v.iloc[0:0] for k,v in dataset['valid_dict'].items()}
    dataset['valid_dict'].update(dataset['train_dict'])

use_train_as_valid(dataset_subsampled)

In [6]:
# Write the subsampled dataset pickle, which can now be used to debug architectures
# by specifying loader_param.datapkl=YOUR_PKL_PATH

dataset_subsampled_path = os.path.join(root_dir, 'rf2aa/subsampled/dataset_20240318_n-2.pkl')
assert not os.path.exists(dataset_subsampled_path)
with open(dataset_subsampled_path, 'wb') as fh:
    pickle.dump(dataset_subsampled, fh)